In [5]:
import os
print(os.path.getsize("train_features.pt"))

157286400


In [8]:
import torch

feature_dict = torch.load("train_features.pt")

train_paths = torch.load("train_paths.pt")
val_paths = torch.load("val_paths.pt")

train_labels = torch.load("train_labels.pt")
val_labels = torch.load("val_labels.pt")

In [10]:
from torch.utils.data import Dataset

class FastDAiSEEDataset(Dataset):
    def __init__(self, paths, labels, feature_dict):
        self.paths = paths
        self.labels = labels
        self.feature_dict = feature_dict

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
      path = self.paths[idx]
      features = self.feature_dict[path]

      # normalize
      features = (features - features.mean()) / (features.std() + 1e-6)

      # temporal dropout
      if torch.rand(1).item() < 0.5:
          mask = torch.rand(features.shape[0]) > 0.1
          features = features[mask]

          if features.shape[0] < 30:
              pad = features[-1].unsqueeze(0).repeat(30 - features.shape[0], 1)
              features = torch.cat([features, pad], dim=0)

      features = features[:30]  # safety

      label = self.labels[idx]
      return features, torch.tensor(label, dtype=torch.long)

In [11]:
from torch.utils.data import DataLoader

train_dataset = FastDAiSEEDataset(train_paths, train_labels, feature_dict)
val_dataset = FastDAiSEEDataset(val_paths, val_labels, feature_dict)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [12]:
f, l = train_dataset[0]
print(f.shape)

torch.Size([30, 1280])


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

from collections import Counter

class_counts = Counter(train_labels)
total = sum(class_counts.values())

class_weights = [(total / class_counts[i])**0.5 for i in range(4)]
class_weights = torch.tensor(class_weights, dtype=torch.float)

# normalize (important)
class_weights = class_weights / class_weights.sum()
class_weights = class_weights.to(device)

In [15]:
import torch.nn as nn

class BiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=1280, hidden_dim=384):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.norm = nn.LayerNorm(hidden_dim * 2)

        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = self.norm(lstm_out)

        attn_context = torch.sum(attn_weights * lstm_out, dim=1)
        mean_context = torch.mean(lstm_out, dim=1)

        context = attn_context + mean_context

        return self.fc(context)

In [16]:
model = BiLSTM_Attention().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5
)

best_acc = 0   # 🔥 ADD BEFORE LOOP

for epoch in range(15):
    model.train()
    train_loss = 0

    for features, labels in train_loader:
        features = features.to(device).float()
        labels = labels.to(device)

        outputs = model(features)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch}, Train Loss: {train_loss / len(train_loader):.4f}")

    # ✅ MOVE VALIDATION HERE
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Validation Accuracy: {acc:.4f}")

    # ✅ SAVE BEST MODEL
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pt")
        print("Saved best model!")

    scheduler.step(acc)

Epoch 0, Train Loss: 1.6272
Validation Accuracy: 0.5896
Saved best model!
Epoch 1, Train Loss: 1.5859
Validation Accuracy: 0.5979
Saved best model!
Epoch 2, Train Loss: 1.5574
Validation Accuracy: 0.5905
Epoch 3, Train Loss: 1.5499
Validation Accuracy: 0.6026
Saved best model!
Epoch 4, Train Loss: 1.5344
Validation Accuracy: 0.6063
Saved best model!
Epoch 5, Train Loss: 1.5237
Validation Accuracy: 0.5989
Epoch 6, Train Loss: 1.5145
Validation Accuracy: 0.5951
Epoch 7, Train Loss: 1.5043
Validation Accuracy: 0.5924
Epoch 8, Train Loss: 1.4942
Validation Accuracy: 0.5933
Epoch 9, Train Loss: 1.4826
Validation Accuracy: 0.6045
Epoch 10, Train Loss: 1.4785
Validation Accuracy: 0.5970
Epoch 11, Train Loss: 1.4734
Validation Accuracy: 0.5998
Epoch 12, Train Loss: 1.4722
Validation Accuracy: 0.5989
Epoch 13, Train Loss: 1.4682
Validation Accuracy: 0.5989
Epoch 14, Train Loss: 1.4674
Validation Accuracy: 0.6007
